In [1]:
import torch
from itertools import combinations

In [2]:
def fit_ols_pairwise(X_ij: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    X = torch.cat(
        [
            torch.ones((X_ij.shape[0], 1), dtype=X_ij.dtype, device=X_ij.device),
            X_ij,
        ],
        dim=1,
    )
    XtX = X.T @ X
    Xty = X.T @ y
    beta =  torch.linalg.lstsq(XtX, Xty.unsqueeze(-1))
    return beta.solution.squeeze(-1)


def all_pairs_explicit_columns(X: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    pairs = list(combinations(range(X.shape[1]), 2))
    out = torch.empty((len(pairs), 3), dtype=X.dtype, device=X.device)
    for k, (i, j) in enumerate(pairs):
        out[k] = fit_ols_pairwise(X[:, [i, j]], y)
    return out


def apply_pairwise_coefficients(X: torch.Tensor, coefs: torch.Tensor) -> torch.Tensor:
    p = X.shape[1]
    pairs = list(combinations(range(p), 2))
    n, c = X.shape[0], len(pairs)
    results = torch.empty((n, c), dtype=X.dtype, device=X.device)
    for k, (i, j) in enumerate(pairs):
        b0, b1, b2 = coefs[k]
        results[:, k] = b0 + b1 * X[:, i] + b2 * X[:, j]
    return results


evaluate = lambda preds, y:  torch.mean(torch.abs(preds - y.unsqueeze(1)), dim=0)

### Входни данни

In [3]:
X = torch.load("X_ep0.pt", weights_only=False)
y = torch.load("y_ep0.pt", weights_only=False)
X.shape, y.shape

(torch.Size([16512, 64]), torch.Size([16512]))

### Хипотези

In [4]:
coef = all_pairs_explicit_columns(X, y)
coef.shape

torch.Size([2016, 3])

In [5]:
preds = apply_pairwise_coefficients(X, coef)
preds.shape

torch.Size([16512, 2016])

### Оценка

In [6]:
errors = evaluate(preds, y)
errors.shape

torch.Size([2016])

### Селекция

In [7]:
mask = torch.argsort(errors)
errors[mask[0]] == errors.min()

tensor(True)

In [8]:
X = preds[:, mask[:100]]
X.shape

torch.Size([16512, 100])

# Chain together 

In [9]:
def epoch(X, y):
    # Хипотези
    coef = all_pairs_explicit_columns(X, y)
    preds = apply_pairwise_coefficients(X, coef)
    # Оценка
    errors = evaluate(preds, y)
    # Селекция
    mask = torch.argsort(errors)
    print(errors.min())
    return preds[:, mask[:100]]

In [10]:
X = torch.load("X_ep0.pt", weights_only=True)
y = torch.load("y_ep0.pt", weights_only=True)
X.shape, y.shape

(torch.Size([16512, 64]), torch.Size([16512]))

In [11]:
for _ in range(5):
    X = epoch(X, y)

tensor(0.5952)
tensor(0.5776)
tensor(0.5707)
tensor(0.5617)
tensor(0.5525)


# Add pow cross

In [12]:
def epoch(X, y, is_pow=False):
    if is_pow:
        X = torch.log(X)
        mask = torch.isfinite(X).all(dim=0)
        X = X[:, mask]

    coef = all_pairs_explicit_columns(X, y)
    preds = apply_pairwise_coefficients(X, coef)

    if is_pow:
        preds = torch.exp(preds)
        y = torch.exp(y)

    errors = evaluate(preds, y)
    mask = torch.argsort(errors)
    return preds[:, mask[:100]], errors[mask[:100]]

In [13]:
X = torch.load("X_ep0.pt", weights_only=False)
y = torch.load("y_ep0.pt", weights_only=False)
y_log = torch.log(y)
X.shape, y.shape, y_log.shape

(torch.Size([16512, 64]), torch.Size([16512]), torch.Size([16512]))

In [ ]:
for _ in range(5):
    X_lin, errors_lin = epoch(X, y)

    X_pow, errors_pow = epoch(X, y_log, is_pow=True)

    X = torch.cat([X_lin, X_pow], dim=1)
    errors = torch.cat([errors_lin, errors_pow], dim=0)
    mask = torch.argsort(errors)
    X = X[:, mask[:100]]
    
    print(f"Lin error: {errors_lin.min():.4f}, Pow error: {errors_pow.min():.4f}, Global error: {errors.min():.4f}")



Lin error: 0.5952, Pow error: 0.5760, Global error: 0.5760
Lin error: 0.5743, Pow error: 0.5649, Global error: 0.5649
Lin error: 0.5600, Pow error: 0.5561, Global error: 0.5561
Lin error: 0.5539, Pow error: 0.5465, Global error: 0.5465
Lin error: 0.5415, Pow error: 0.5348, Global error: 0.5348
